In [ ]:
import requests
import pandas as pd
from io import StringIO

url = "https://raw.githubusercontent.com/MohammedHameds/test1455/refs/heads/main/products.csv"

response = requests.get(url)

products = pd.read_csv(StringIO(response.text))

products.to_csv('products.csv', index=False)

### Customers

In [ ]:
customers = pd.read_csv('customers.csv')
customers

In [ ]:
customers = customers.drop_duplicates(subset=["first_name", "last_name", "email"])
customers.duplicated()

In [ ]:
customers['city'] = customers['city'].str.strip().str.title()

In [ ]:
customers["age"] = pd.to_numeric(customers["age"], errors="coerce")
customers = customers[(customers["age"] >= 5) & (customers["age"] <= 100)]

In [ ]:
customers["email"] = customers["email"].fillna("unknown@email.com")
customers["email"].isna().sum()


In [ ]:
customers['full_name'] = customers['first_name'] + ' ' + customers['last_name']
customers

In [ ]:
customers.to_csv("clean_customers.csv", index=False)

### Orders

In [ ]:
orders = pd.read_parquet("orders.parquet")
orders

In [ ]:
orders['order_id'].duplicated().sum()

In [ ]:
orders = orders.drop_duplicates(subset="order_id")
orders['order_id'].duplicated().sum()

In [ ]:
orders["quantity"] = pd.to_numeric(orders["quantity"], errors="coerce")
orders = orders.dropna(subset=["quantity"])
orders = orders[orders["quantity"] > 0]

In [ ]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

In [ ]:
orders["customer_id"] = orders["customer_id"].replace({5: 1})
orders = orders[orders["customer_id"].isin(customers["customer_id"])]

In [ ]:
orders.to_csv("cleaned_orders.csv", index=False)

### Products

In [ ]:
products = pd.read_csv('products.csv')
products

In [ ]:
products = products.drop_duplicates()

In [ ]:
products["product_name"] = products["product_name"].str.strip().str.title()
products["category"] = products["category"].str.strip().str.title()

In [ ]:
products['unit_price'] = pd.to_numeric(products['unit_price'], errors= "coerce")
products = products.dropna(subset=["unit_price"])
products = products[products["unit_price"] > 0]

In [ ]:
products.to_csv('cleaned_products.csv', index=False)

### Mergeing

In [ ]:
sales = orders.merge(customers, on="customer_id", how="inner")
sales = sales.merge(products, on="product_id", how="inner")

sales

In [ ]:
sales["total_amount"] = sales["quantity"] * sales["unit_price"]

sales

In [ ]:
sales = sales.rename(columns={"full_name": "customer", "product_name": "product"})
sales = sales[["order_id", "customer", "product", "quantity", "unit_price", "total_amount"]]

In [ ]:
sales.to_csv("sales.csv", index=False)